# Statistical Learning 2 — Practice Test Solutions
**FH Kufstein Tirol | SS 2026 | 16. May 2026**

---

This notebook provides **complete, exam-ready answers** to all exercises in the Probeklausur (Practice Test), including visualizations that illustrate each key concept.

| Exercise | Topic | Points |
|----------|-------|--------|
| 1 | General Linear Regression & Basis Functions | 4 |
| 2 | Bias-Variance Decomposition | 6 |
| 3 | Regularization (Tikhonov, Early Stopping) | 5 |
| 4 | Convexity & Gradient Descent with Nesterov | 5 |
| **Total** | | **20** |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams.update({'font.size': 11, 'figure.dpi': 110})

---
## Exercise 1 — General Linear Regression (4 points)

> Let $\{(x_1,y_1),(x_2,y_2),\ldots,(x_n,y_n)\} \subset \mathbb{R}^2$ be a dataset.
> We are considering the regression problem of finding $f$ such that $f(x_i) \approx y_i$.

### (a) Model ansatz for general linear regression (1 point)

**Answer:**

Given basis functions $B_1, \ldots, B_p : \mathbb{R} \to \mathbb{R}$, the model ansatz for **general linear regression** is:

$$\boxed{f(x) = \sum_{j=1}^{p} \theta_j B_j(x) = \mathbf{B}(x)^\top \boldsymbol{\theta}}$$

where:
- $\boldsymbol{\theta} = (\theta_1, \theta_2, \ldots, \theta_p)^\top \in \mathbb{R}^p$ is the **parameter vector**
- $\mathbf{B}(x) = (B_1(x), B_2(x), \ldots, B_p(x))^\top$ is the vector of evaluated basis functions
- The model is **linear in the parameters** $\boldsymbol{\theta}$ (but may be nonlinear in $x$)

**Examples of basis functions:**
- Polynomial: $B_j(x) = x^{j-1}$ → $f(x) = \theta_1 + \theta_2 x + \theta_3 x^2 + \ldots$
- Fourier: $B_j(x) = \sin(jx)$ or $\cos(jx)$
- Radial basis functions: $B_j(x) = \exp(-\|x - c_j\|^2)$

### (b) Which components are fitted to the data? (1 point)

**Answer:**

Only the **parameters** $\boldsymbol{\theta} = (\theta_1, \ldots, \theta_p)^\top$ **are fitted** (optimized) to the data.

The basis functions $B_1, \ldots, B_p$ are **fixed a priori** (chosen by the modeler before training) and are **not** learned from the data. This is the key property that makes the model "linear" — the optimization problem over $\boldsymbol{\theta}$ is a linear least squares problem.

> **Key insight:** The choice of basis functions encodes our prior knowledge about the structure of $f$. Once chosen, only $\boldsymbol{\theta}$ is estimated from data.

### (c) Normal equation — finding optimal parameters (2 points)

**Answer:**

We minimize the **sum of squared residuals** (Ordinary Least Squares):

$$\mathcal{L}(\boldsymbol{\theta}) = \sum_{i=1}^n \left(y_i - f(x_i)\right)^2 = \|\mathbf{y} - \mathbf{B}\boldsymbol{\theta}\|^2$$

where the **design matrix** $\mathbf{B} \in \mathbb{R}^{n \times p}$ has entries $\mathbf{B}_{ij} = B_j(x_i)$.

Setting $\nabla_{\boldsymbol{\theta}} \mathcal{L} = 0$:

$$-2\mathbf{B}^\top(\mathbf{y} - \mathbf{B}\boldsymbol{\theta}) = 0 \implies \mathbf{B}^\top \mathbf{B}\,\boldsymbol{\theta} = \mathbf{B}^\top \mathbf{y}$$

This is the **Normal Equation**. If $\mathbf{B}^\top\mathbf{B}$ is invertible:

$$\boxed{\boldsymbol{\theta}^* = (\mathbf{B}^\top \mathbf{B})^{-1} \mathbf{B}^\top \mathbf{y}}$$

This is the **closed-form (analytic) solution** — no iterative optimization needed.

**Note:** $\mathbf{B}^\top\mathbf{B}$ is invertible when the columns of $\mathbf{B}$ are linearly independent (i.e., $n \geq p$ and no basis function is a linear combination of others).

In [ ]:
# === Visualization: Basis functions and fitted model (polynomial example) ===

# True function + noisy data
x_true = np.linspace(0, 2*np.pi, 300)
f_true = np.sin(x_true)

np.random.seed(0)
x_data = np.sort(np.random.uniform(0, 2*np.pi, 20))
y_data = np.sin(x_data) + np.random.normal(0, 0.2, 20)

# Build design matrix with polynomial basis B_j(x) = x^(j-1)
def design_matrix(x, p):
    return np.column_stack([x**j for j in range(p)])

# Fit via normal equation
def fit_normal_eq(x, y, p):
    B = design_matrix(x, p)
    theta = np.linalg.solve(B.T @ B, B.T @ y)
    return theta

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Left: Show first 5 polynomial basis functions
ax = axes[0]
colors_bf = ['#E74C3C','#3498DB','#2ECC71','#F39C12','#9B59B6']
x_plot = np.linspace(0, 2*np.pi, 200)
for j, c in enumerate(colors_bf):
    ax.plot(x_plot, x_plot**j / max(1, (2*np.pi)**j), color=c, lw=2,
            label=f'$B_{j+1}(x) = x^{j}$' if j > 0 else '$B_1(x) = 1$')
ax.set_title('Basis Functions $B_1,\\ldots,B_5$ (polynomial)', fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('$B_j(x)$ (normalized)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Middle: Design matrix heatmap
ax = axes[1]
B_vis = design_matrix(x_data, 5)
B_norm = (B_vis - B_vis.min(axis=0)) / (B_vis.max(axis=0) - B_vis.min(axis=0) + 1e-8)
im = ax.imshow(B_norm, aspect='auto', cmap='Blues')
ax.set_title('Design Matrix $\\mathbf{B} \\in \\mathbb{R}^{n \\times p}$\n(n=20 samples, p=5 basis functions)',
             fontweight='bold')
ax.set_xlabel('Basis function index j'); ax.set_ylabel('Sample index i')
ax.set_xticks(range(5)); ax.set_xticklabels([f'$B_{j+1}$' for j in range(5)])
plt.colorbar(im, ax=ax, label='Value (normalized)')

# Right: Normal equation fit with different p
ax = axes[2]
ax.scatter(x_data, y_data, color='black', s=40, zorder=5, label='Data $(x_i, y_i)$')
ax.plot(x_true, f_true, 'k--', lw=1.5, alpha=0.5, label='True $f(x)=\\sin(x)$')

for p, color, ls in [(2,'#E74C3C','-'), (5,'#3498DB','-'), (10,'#2ECC71','-')]:
    theta = fit_normal_eq(x_data, y_data, p)
    B_pred = design_matrix(x_true, p)
    y_pred = B_pred @ theta
    ax.plot(x_true, y_pred, color=color, lw=2, ls=ls, label=f'p={p} basis functions')

ax.set_title('Normal Equation Fit: $\\theta^* = (\\mathbf{B}^\\top\\mathbf{B})^{-1}\\mathbf{B}^\\top\\mathbf{y}$',
             fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.set_ylim(-2, 2)

plt.suptitle('Exercise 1 — General Linear Regression & Normal Equation', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Exercise 2 — Bias-Variance Decomposition (6 points)

### (a) Bias-Variance decomposition — formula, intuition, and relation to p (2 points)

**Answer:**

The **expected Mean Squared Error** at a point $x$ can be decomposed as:

$$\mathbb{E}\left[(y - \hat{f}(x))^2\right] = \underbrace{\left(f(x) - \mathbb{E}[\hat{f}(x)]\right)^2}_{\text{Bias}^2} + \underbrace{\mathbb{E}\left[\left(\hat{f}(x) - \mathbb{E}[\hat{f}(x)]\right)^2\right]}_{\text{Variance}} + \underbrace{\sigma_\varepsilon^2}_{\text{Irreducible Noise}}$$

**Intuition of each term:**
- **Bias²:** Systematic error — how far the *average* prediction is from the truth. A model that is too simple (few basis functions) cannot capture the true shape of $f$ → high bias.
- **Variance:** Sensitivity to the specific training set — how much $\hat{f}$ fluctuates across different datasets. A model that is too complex (many basis functions) fits noise → high variance.
- **Irreducible noise ($\sigma_\varepsilon^2$):** The noise in the data itself — cannot be reduced by any model.

**Relation to the number of basis functions $p$:**

| $p$ small | $p$ large |
|-----------|----------|
| High Bias (underfitting) | Low Bias (can fit complex shapes) |
| Low Variance (stable predictions) | High Variance (fits noise) |

> **The bias-variance trade-off:** increasing $p$ decreases bias but increases variance. The optimal $p$ minimizes total MSE = Bias² + Variance + const.

### (b) At least 3 methods to reduce variance (3 points)

**Answer — 6 methods with explanations:**

1. **Regularization (L2 / Ridge / Tikhonov):** Add penalty $\lambda\|\boldsymbol{\theta}\|^2$ to the loss. This shrinks coefficients toward zero, preventing the model from fitting noise. Higher $\lambda$ → lower variance, higher bias.

2. **Regularization (L1 / Lasso):** Add penalty $\lambda\|\boldsymbol{\theta}\|_1$. Produces sparse solutions (sets some $\theta_j = 0$ exactly), performing automatic feature selection → reduces effective model complexity → reduces variance.

3. **Early Stopping:** In iterative optimization (e.g., gradient descent), stop training before convergence. The early iterate acts as a regularizer — fewer update steps → less overfitting to training data → reduced variance.

4. **Bagging (Bootstrap Aggregating):** Train $B$ models on bootstrap samples of the data, average their predictions: $\hat{f}_{bag}(x) = \frac{1}{B}\sum_{b=1}^B \hat{f}^{(b)}(x)$. Averaging over $B$ independent estimators reduces variance by a factor of $B$.

5. **Reducing model complexity (fewer basis functions):** Use fewer basis functions $p$ → simpler model → lower variance. Choose $p$ via cross-validation.

6. **Dropout (for neural networks):** Randomly zero out neurons during training. Acts as ensemble averaging over many sub-networks → reduces variance.

### (c) Model with zero variance — describe and comment on usefulness (1 point)

**Answer:**

A model with **zero variance** is a model whose predictions do not depend on the training data at all. The canonical example is the **constant model**:

$$\hat{f}(x) = c \quad \text{(e.g., } c = \bar{y} = \frac{1}{n}\sum_{i=1}^n y_i \text{)}$$

**Zero variance:** for any two training sets $\mathcal{D}_1, \mathcal{D}_2$, $\hat{f}_{\mathcal{D}_1}(x) = \hat{f}_{\mathcal{D}_2}(x)$ → predictions are perfectly stable.

**Usefulness:** A constant model is essentially **useless** for any real regression task:
- It has **maximum bias** — it ignores all input features $x$ completely
- It can only predict the training mean, regardless of the input
- It serves as a **baseline** ("dummy regressor") to compare against

> Zero variance does not mean low error — it just means the model is so inflexible that it cannot learn anything from data.

In [ ]:
# === Visualization: Bias-Variance Trade-off ===

np.random.seed(42)

def true_f(x):
    return np.sin(x * 2) + 0.5 * x

x_all = np.linspace(0, 3, 300)
n_datasets = 50
n_pts = 25
sigma_noise = 0.4

# Use Ridge with tiny lambda for numerical stability on high-degree polys
def fit_ridge(x_tr, y_tr, p, lam=1e-6):
    B = np.column_stack([x_tr**j for j in range(p+1)])
    theta = np.linalg.solve(B.T @ B + lam * np.eye(p+1), B.T @ y_tr)
    return theta

degrees = [1, 4, 10]
preds_all = {d: np.zeros((n_datasets, len(x_all))) for d in degrees}

for trial in range(n_datasets):
    x_tr = np.sort(np.random.uniform(0, 3, n_pts))
    y_tr = true_f(x_tr) + np.random.normal(0, sigma_noise, n_pts)
    for d in degrees:
        B_te = np.column_stack([x_all**j for j in range(d+1)])
        theta = fit_ridge(x_tr, y_tr, d)
        pred = B_te @ theta
        preds_all[d][trial] = np.clip(pred, -5, 8)  # clip extremes from unstable polys

labels = {1: 'p=2  (High Bias, Low Variance)',
          4: 'p=5  (Good balance)',
          10: 'p=11 (Low Bias, High Variance)'}
colors_d = {1: '#E74C3C', 4: '#2ECC71', 10: '#3498DB'}

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for col, d in enumerate(degrees):
    ax = axes[0, col]
    mean_pred = preds_all[d].mean(axis=0)
    std_pred  = preds_all[d].std(axis=0)
    for t in range(min(15, n_datasets)):
        ax.plot(x_all, preds_all[d][t], alpha=0.15, color=colors_d[d], lw=1)
    ax.plot(x_all, mean_pred, color=colors_d[d], lw=2.5, label='Mean prediction')
    ax.fill_between(x_all, mean_pred - std_pred, mean_pred + std_pred,
                    alpha=0.25, color=colors_d[d], label='± 1 std')
    ax.plot(x_all, true_f(x_all), 'k--', lw=2, label='True f(x)')
    ax.set_title(labels[d], fontweight='bold', fontsize=10)
    ax.set_xlabel('x'); ax.set_ylabel('f(x)')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.set_ylim(-2.5, 3.5)

# Bias², Variance, MSE vs p
ps = list(range(1, 12))
bias2s, vars_, mses = [], [], []
preds_p = {p: np.zeros((n_datasets, len(x_all))) for p in ps}

for trial in range(n_datasets):
    x_tr = np.sort(np.random.uniform(0, 3, n_pts))
    y_tr = true_f(x_tr) + np.random.normal(0, sigma_noise, n_pts)
    for p in ps:
        B_te = np.column_stack([x_all**j for j in range(p+1)])
        theta = fit_ridge(x_tr, y_tr, p)
        pred = B_te @ theta
        preds_p[p][trial] = np.clip(pred, -5, 8)

for p in ps:
    mean_p = preds_p[p].mean(axis=0)
    bias2 = float(np.mean((mean_p - true_f(x_all))**2))
    var   = float(np.mean(preds_p[p].var(axis=0)))
    bias2s.append(bias2); vars_.append(var)
    mses.append(bias2 + var + sigma_noise**2)

ax = axes[1, 0]
ax.plot(ps, bias2s, 'r-o', ms=5, lw=2, label='Bias²')
ax.plot(ps, vars_,  'b-s', ms=5, lw=2, label='Variance')
ax.plot(ps, mses,   'k-^', ms=5, lw=2.5, label='Total MSE')
ax.axhline(sigma_noise**2, color='gray', ls=':', lw=1.5,
           label=f'Irreducible noise σ²={sigma_noise**2:.2f}')
best_p = ps[int(np.argmin(mses))]
ax.axvline(best_p, color='green', ls='--', lw=1.5, label=f'Optimal p={best_p}')
ax.set_title('Bias²-Variance Trade-off vs p', fontweight='bold')
ax.set_xlabel('Number of basis functions p')
ax.set_ylabel('Error')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Zero-variance model
ax = axes[1, 1]
x_demo = np.sort(np.random.uniform(0, 3, 30))
y_demo = true_f(x_demo) + np.random.normal(0, 0.4, 30)
const_pred = np.mean(y_demo)
ax.scatter(x_demo, y_demo, color='black', s=30, label='Data', zorder=4)
ax.plot(x_all, true_f(x_all), 'k--', lw=2, label='True f(x)')
ax.axhline(const_pred, color='red', lw=2.5,
           label=f'Constant model = ȳ = {const_pred:.2f}\n(Zero variance, Max bias)')
ax.set_title('Zero-Variance Model: $\\hat{f}(x) = \\bar{y}$\n(Constant — useless for regression)',
             fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Variance reduction comparison (illustrative)
ax = axes[1, 2]
methods = ['OLS\np=11\n(no reg.)', 'Ridge\n(L2)\nλ=1', 'Lasso\n(L1)\nλ=0.1', 'Bagging\nB=30', 'Constant\n(ȳ)']
bias2_m = [0.08, 0.12, 0.10, 0.07, 0.95]
var_m   = [0.82, 0.15, 0.18, 0.12, 0.00]
x_m = np.arange(len(methods))
width = 0.35
ax.bar(x_m - width/2, bias2_m, width, label='Bias²',    color='#E74C3C', alpha=0.85)
ax.bar(x_m + width/2, var_m,   width, label='Variance', color='#3498DB', alpha=0.85)
ax.set_title('Bias² vs Variance across methods\n(illustrative values)', fontweight='bold')
ax.set_xticks(x_m); ax.set_xticklabels(methods, fontsize=8)
ax.set_ylabel('Error component')
ax.legend(); ax.grid(alpha=0.3, axis='y')

plt.suptitle('Exercise 2 — Bias-Variance Decomposition', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Exercise 3 — Regularization (5 points)

### (a) Purpose of regularization in machine learning (1 point)

**Answer:**

The purpose of **regularization** is to **prevent overfitting** (reduce variance) by adding additional constraints or penalty terms to the optimization problem.

Without regularization, a complex model (large $p$) minimizes training error perfectly but generalizes poorly to new data. Regularization introduces a **bias toward simpler solutions**, trading some training accuracy for better generalization.

Formally, instead of minimizing only the data-fit term, we minimize:
$$\mathcal{L}_{\text{reg}}(\boldsymbol{\theta}) = \underbrace{\mathcal{L}_{\text{data}}(\boldsymbol{\theta})}_{\text{fit data}} + \lambda \underbrace{\mathcal{R}(\boldsymbol{\theta})}_{\text{regularizer}}$$

The hyperparameter $\lambda \geq 0$ controls the **strength of regularization**: $\lambda = 0$ → no regularization; $\lambda \to \infty$ → maximum regularization (trivial solution).

### (b) Variational Tikhonov Regularization — functional and loss function (3 points)

**Answer:**

**Variational Tikhonov regularization** (also known as Ridge regression in the linear case) formulates the learning problem as:

$$\boxed{\hat{f} = \arg\min_{f \in \mathcal{H}} \left[ \underbrace{\frac{1}{n}\sum_{i=1}^n \ell(f(x_i), y_i)}_{\text{Data Fit / Empirical Loss}} + \lambda \underbrace{\|\Gamma f\|^2}_{\text{Regularization Functional}} \right]}$$

**Components:**

1. **Loss function** $\ell(f(x_i), y_i)$: measures how well $f$ fits each data point. For regression:
   - Squared loss: $\ell(\hat{y}, y) = (\hat{y} - y)^2$ → leads to MSE as empirical loss

2. **Regularization functional** $\|\Gamma f\|^2$: penalizes undesired properties of $f$, controlled by the **Tikhonov operator** $\Gamma$:
   - $\Gamma = I$ (identity) → $\|\boldsymbol{\theta}\|^2$ = **L2 / Ridge**: penalizes large parameters
   - $\Gamma = D^k$ (differential operator) → $\|f^{(k)}\|^2$: penalizes roughness (curvature of $f$)
   - $\Gamma = $ arbitrary matrix → general Tikhonov regularization

**In the linear model case** ($f(x) = \mathbf{B}(x)^\top \boldsymbol{\theta}$, $\Gamma = I$):

$$\min_{\boldsymbol{\theta}} \|\mathbf{B}\boldsymbol{\theta} - \mathbf{y}\|^2 + \lambda\|\boldsymbol{\theta}\|^2$$

This has a **closed-form solution** (Ridge regression):

$$\boldsymbol{\theta}^*_{\text{Ridge}} = (\mathbf{B}^\top\mathbf{B} + \lambda I)^{-1}\mathbf{B}^\top\mathbf{y}$$

The $\lambda I$ term ensures the matrix is always invertible (even when $\mathbf{B}^\top\mathbf{B}$ is singular, i.e., $p > n$).

### (c) Regularization by Early Stopping (1 point)

**Answer:**

**Basic principle:** In iterative optimization (e.g., gradient descent), the model complexity effectively *grows* with the number of iterations. Early stopping halts training when the **validation error stops improving**, before the model has fully converged to the training minimum.

**Why it regularizes:** At the start of training, the model makes large, coarse updates (fits the main structure of the data). As training continues, it begins fitting smaller details — eventually fitting noise (overfitting). Stopping early prevents this last phase.

**Algorithm:**
```
Initialize θ₀
best_val_loss = ∞,  patience_counter = 0
for t = 1, 2, ..., T_max:
    θₜ = θₜ₋₁ - η ∇L_train(θₜ₋₁)        # gradient step
    val_loss = L_val(θₜ)                   # evaluate on validation set
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_θ = θₜ
        patience_counter = 0
    else:
        patience_counter += 1
    if patience_counter >= patience:
        break                              # early stop!
return best_θ
```

> **Connection to Tikhonov:** For linear models trained with gradient descent, early stopping is approximately equivalent to L2 regularization with $\lambda \propto 1/t$.

In [ ]:
# === Visualization: Regularization ===

np.random.seed(5)
n = 30
x_r = np.sort(np.random.uniform(0, 3, n))
y_r = np.sin(2*x_r) + 0.5*x_r + np.random.normal(0, 0.35, n)
x_plot = np.linspace(0, 3, 300)

# Normalize x to [0,1] for numerical stability with high-degree polynomials
x_r_n = x_r / 3.0
x_plot_n = x_plot / 3.0

def poly_design(x, p):
    return np.column_stack([x**j for j in range(p)])

p = 12
B_tr = poly_design(x_r_n, p)
B_pl = poly_design(x_plot_n, p)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: OLS vs Ridge (Tikhonov)
ax = axes[0]
ax.scatter(x_r, y_r, color='black', s=30, zorder=5, label='Training data')
ax.plot(x_plot, np.sin(2*x_plot) + 0.5*x_plot, 'k--', lw=1.5, alpha=0.5, label='True f(x)')

for lam, color, name in [(0, '#E74C3C', 'OLS (λ=0)'),
                          (1e-4, '#3498DB', 'Ridge (λ=1e-4)'),
                          (1e-2, '#2ECC71', 'Ridge (λ=1e-2)')]:
    theta = np.linalg.solve(B_tr.T @ B_tr + lam * np.eye(p), B_tr.T @ y_r)
    pred = np.clip(B_pl @ theta, -3, 4)
    ax.plot(x_plot, pred, color=color, lw=2, label=name)

ax.set_title('Tikhonov (Ridge) Regularization\n'
             '$\\min \\|\\mathbf{B}\\theta - y\\|^2 + \\lambda\\|\\theta\\|^2$', fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.set_ylim(-3, 4)

# Plot 2: Ridge coefficient path
ax = axes[1]
lambdas = np.logspace(-6, 1, 100)
coef_paths = []
for lam in lambdas:
    theta = np.linalg.solve(B_tr.T @ B_tr + lam * np.eye(p), B_tr.T @ y_r)
    coef_paths.append(theta)
coef_paths = np.array(coef_paths)

for j in range(p):
    ax.semilogx(lambdas, coef_paths[:, j], lw=1.5, alpha=0.8)
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.axvline(1e-4, color='blue', ls=':', lw=2, label='λ=1e-4')
ax.set_title('Ridge Regularization Path\n(Coefficients shrink toward 0 as λ→∞)', fontweight='bold')
ax.set_xlabel('λ (log scale)'); ax.set_ylabel('Coefficient value $\\theta_j$')
ax.legend(); ax.grid(alpha=0.3)
ax.text(0.02, 0.95, 'λ→0: OLS\n(high variance)', transform=ax.transAxes,
        va='top', fontsize=9, color='#E74C3C')
ax.text(0.72, 0.95, 'λ→∞: θ≈0\n(high bias)', transform=ax.transAxes,
        va='top', fontsize=9, color='#2ECC71')

# Plot 3: Early stopping — use sklearn Ridge with decreasing alpha to simulate
from sklearn.linear_model import Ridge as RidgeSK
ax = axes[2]

# Simulate early stopping: decreasing lambda ↔ more training iterations
lambdas_es = np.logspace(2, -4, 150)   # from strong reg → no reg (like training epochs)
val_x = np.linspace(0.1, 2.9, 50)
val_x_n = val_x / 3.0
y_val_true = np.sin(2*val_x) + 0.5*val_x
y_val_noisy = y_val_true + np.random.normal(0, 0.35, len(val_x))

B_val = poly_design(val_x_n, p)

train_losses_es, val_losses_es = [], []
for lam in lambdas_es:
    theta = np.linalg.solve(B_tr.T @ B_tr + lam * np.eye(p), B_tr.T @ y_r)
    train_losses_es.append(float(np.mean((y_r - B_tr @ theta)**2)))
    val_losses_es.append(float(np.mean((y_val_noisy - B_val @ theta)**2)))

train_losses_es = np.array(train_losses_es)
val_losses_es   = np.array(val_losses_es)
epochs = np.arange(1, len(lambdas_es) + 1)
best_ep = int(np.argmin(val_losses_es))

ax.plot(epochs, train_losses_es, 'b-', lw=2, label='Training loss')
ax.plot(epochs, val_losses_es,   'r-', lw=2, label='Validation loss')
ax.axvline(best_ep + 1, color='green', ls='--', lw=2,
           label=f'Early stop (step {best_ep+1})')
ax.scatter([best_ep + 1], [val_losses_es[best_ep]], s=100, color='green', zorder=5)
y_top = float(np.percentile(val_losses_es, 85))
ax.fill_betweenx([0, y_top], best_ep + 1, len(lambdas_es),
                 alpha=0.1, color='red', label='Overfitting zone')
ax.set_title('Early Stopping — training proceeds\nfrom over-regularized → overfitted', fontweight='bold')
ax.set_xlabel('Training step (decreasing λ)'); ax.set_ylabel('MSE Loss')
ax.set_ylim(0, y_top)
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle('Exercise 3 — Regularization: Tikhonov / Ridge & Early Stopping',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Exercise 4 — Convexity & Gradient Descent with Nesterov (5 points)

> Let $\mathcal{L}: \mathbb{R}^d \to \mathbb{R}$, $\theta \mapsto \mathcal{L}(\theta)$ be a function for which we want to compute the minimizer.

### (a) Convexity in optimization — local vs global minimizers (2 points)

**Answer:**

**Definition of convexity:**
A function $\mathcal{L}: \mathbb{R}^d \to \mathbb{R}$ is **convex** if for all $\theta_1, \theta_2 \in \mathbb{R}^d$ and all $\lambda \in [0,1]$:

$$\mathcal{L}(\lambda\,\theta_1 + (1-\lambda)\,\theta_2) \leq \lambda\,\mathcal{L}(\theta_1) + (1-\lambda)\,\mathcal{L}(\theta_2)$$

Geometrically: the function lies **below or on the chord** connecting any two points on its graph.

**Why convexity is important for optimization:**

1. **Every local minimum is a global minimum:** For a convex $\mathcal{L}$, any point $\theta^*$ satisfying $\nabla\mathcal{L}(\theta^*) = 0$ is a **global minimizer**. This means gradient descent (and any first-order method) is **guaranteed to find the global optimum**.

2. **Non-convex functions** can have many local minima, saddle points, and flat regions. Gradient descent may get stuck in a local minimum that is far from the global optimum.

3. **Practical consequence:** Many ML loss functions are convex (linear regression with MSE, logistic regression, SVMs, Ridge regression). For these, optimization is well-posed and solved efficiently.

**Conclusion about local and global minimizers for convex $\mathcal{L}$:**

> If $\mathcal{L}$ is **convex**, then every **local minimizer is also a global minimizer**. The set of global minimizers is a **convex set**. If $\mathcal{L}$ is **strictly convex**, the global minimizer is **unique**.

### (b) Gradient Descent with Nesterov Acceleration — pseudo-code (3 points)

**Answer:**

**Standard Gradient Descent** (for reference):
```
Initialize: θ₀
for t = 0, 1, 2, ..., T:
    θₜ₊₁ = θₜ - η ∇L(θₜ)
```

**Nesterov Accelerated Gradient Descent (NAG):**

The key idea: instead of computing the gradient at the **current** position $\theta_t$, compute it at a **look-ahead** position $y_t = \theta_t + \text{momentum}$.

```
Initialize: θ₀, y₁ = θ₀,  λ₀ = 1
for t = 1, 2, ..., T:
    
    // Gradient step at the look-ahead point yₜ
    θₜ = yₜ - η ∇L(yₜ)
    
    // Update momentum coefficient
    λₜ₊₁ = (1 + sqrt(1 + 4λₜ²)) / 2
    
    // Compute next look-ahead point (extrapolation)
    γₜ = (λₜ - 1) / λₜ₊₁
    yₜ₊₁ = θₜ + γₜ · (θₜ - θₜ₋₁)

return θ_T
```

**Simplified version (fixed momentum $\mu$):**
```
Initialize: θ₀, v₀ = 0,  μ ∈ (0,1)  (e.g. μ = 0.9)
for t = 0, 1, 2, ..., T:
    
    // Look-ahead: compute gradient at θₜ + μ·vₜ
    look_ahead = θₜ + μ · vₜ
    g = ∇L(look_ahead)
    
    // Update velocity (momentum)
    vₜ₊₁ = μ · vₜ - η · g
    
    // Update parameters
    θₜ₊₁ = θₜ + vₜ₊₁

return θ_T
```

**Why Nesterov is better than standard GD:**
- Standard GD: convergence rate $O(1/t)$ for convex functions
- Nesterov: convergence rate $O(1/t^2)$ — **optimal** for first-order methods on convex functions
- The look-ahead step corrects the momentum direction *before* it overshoots, leading to much faster convergence

In [ ]:
# === Visualization: Convexity & Gradient Descent with Nesterov ===

fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)

# ---- Panel 1: Convex vs Non-Convex ----
ax1 = fig.add_subplot(gs[0, 0])
x_cv = np.linspace(-3, 3, 300)
convex_f = x_cv**2 + 1
nonconv_f = x_cv**4 - 3*x_cv**2 + x_cv + 2

ax1.plot(x_cv, convex_f, 'b-', lw=2.5, label='Convex: $f(x)=x^2+1$')
# Chord for convex
xa, xb = -2, 1.5
ax1.plot([xa, xb], [xa**2+1, xb**2+1], 'b--', lw=1.5, alpha=0.5)
ax1.scatter([0], [1], s=120, color='blue', zorder=5, label='Global min = Local min')
ax1.text(0.1, 1.3, 'Global\nmin', color='blue', fontsize=8)

ax1.plot(x_cv, nonconv_f, 'r-', lw=2.5, label='Non-convex: $x^4-3x^2+x+2$')
local_mins_x = [-1.35, 1.28]
for lx in local_mins_x:
    ax1.scatter([lx], [lx**4 - 3*lx**2 + lx + 2], s=100, color='red', zorder=5)
ax1.annotate('Local\nmin', (-1.35, -0.3), (-2.2, 0.5), fontsize=8, color='red',
             arrowprops=dict(arrowstyle='->', color='red'))
ax1.annotate('Global\nmin', (1.28, -0.5), (1.8, 0.5), fontsize=8, color='darkred',
             arrowprops=dict(arrowstyle='->', color='darkred'))

ax1.set_title('Convex vs Non-Convex Functions', fontweight='bold')
ax1.set_xlabel('θ'); ax1.set_ylabel('L(θ)')
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)
ax1.set_ylim(-2, 12)

# ---- Panel 2: Convexity definition (chord) ----
ax2 = fig.add_subplot(gs[0, 1])
x_def = np.linspace(-2, 2, 200)
f_def = x_def**2 + 0.3
ax2.plot(x_def, f_def, 'b-', lw=3, label='Convex $\\mathcal{L}(\\theta)$')

t1, t2, lam = -1.5, 1.2, 0.4
f1, f2 = t1**2+0.3, t2**2+0.3
t_mid = lam*t1 + (1-lam)*t2
f_chord = lam*f1 + (1-lam)*f2
f_func  = t_mid**2 + 0.3

ax2.plot([t1, t2], [f1, f2], 'g--', lw=2, label='Chord')
ax2.scatter([t1, t2], [f1, f2], s=80, color='green', zorder=5)
ax2.scatter([t_mid], [f_chord], s=100, color='green', marker='^', zorder=5, label=f'Chord: λf(θ₁)+(1-λ)f(θ₂)')
ax2.scatter([t_mid], [f_func], s=100, color='blue', marker='v', zorder=5, label=f'f(λθ₁+(1-λ)θ₂)')
ax2.annotate('', xy=(t_mid, f_func), xytext=(t_mid, f_chord),
             arrowprops=dict(arrowstyle='<->', color='red', lw=2))
ax2.text(t_mid+0.08, (f_func+f_chord)/2, 'f ≤ chord\n(convex!)', color='red', fontsize=8)
ax2.set_title('Convexity Definition:\n$\\mathcal{L}(\\lambda\\theta_1+(1-\\lambda)\\theta_2) \\leq \\lambda\\mathcal{L}(\\theta_1)+(1-\\lambda)\\mathcal{L}(\\theta_2)$',
              fontweight='bold', fontsize=9)
ax2.set_xlabel('θ'); ax2.set_ylabel('L(θ)')
ax2.legend(fontsize=8, loc='upper center'); ax2.grid(alpha=0.3)

# ---- Panel 3: 2D loss landscape — GD vs Nesterov paths ----
ax3 = fig.add_subplot(gs[0, 2])

# 2D quadratic (convex)
t1_g = np.linspace(-4, 4, 200)
t2_g = np.linspace(-3, 3, 200)
T1, T2 = np.meshgrid(t1_g, t2_g)
L2D = 2*T1**2 + 5*T2**2  # elongated — tests momentum advantage
ax3.contour(T1, T2, L2D, levels=20, cmap='Blues', alpha=0.6)
ax3.contourf(T1, T2, L2D, levels=20, cmap='Blues', alpha=0.2)

# GD path
def grad_L(t): return np.array([4*t[0], 10*t[1]])

eta_gd = 0.09
theta = np.array([3.5, 2.5])
path_gd = [theta.copy()]
for _ in range(40):
    theta = theta - eta_gd * grad_L(theta)
    path_gd.append(theta.copy())
path_gd = np.array(path_gd)

# Nesterov path
mu = 0.85
theta = np.array([3.5, 2.5])
v = np.zeros(2)
path_nes = [theta.copy()]
for _ in range(40):
    look = theta + mu * v
    v = mu * v - eta_gd * grad_L(look)
    theta = theta + v
    path_nes.append(theta.copy())
path_nes = np.array(path_nes)

ax3.plot(path_gd[:,0], path_gd[:,1], 'r-o', ms=3, lw=1.5, label=f'GD ({len(path_gd)-1} steps)')
ax3.plot(path_nes[:,0], path_nes[:,1], 'g-s', ms=3, lw=1.5, label=f'Nesterov ({len(path_nes)-1} steps)')
ax3.scatter([3.5], [2.5], s=100, color='black', zorder=6, marker='*', label='Start')
ax3.scatter([0], [0], s=150, color='gold', edgecolor='black', zorder=6, marker='*', label='Optimum')
ax3.set_title('GD vs Nesterov Trajectories\n(2D convex quadratic)', fontweight='bold')
ax3.set_xlabel('θ₁'); ax3.set_ylabel('θ₂')
ax3.legend(fontsize=9); ax3.grid(alpha=0.3)

# ---- Panel 4: Loss convergence comparison ----
ax4 = fig.add_subplot(gs[1, 0])

def run_gd(n_steps, eta):
    theta = np.array([3.5, 2.5])
    losses = [2*theta[0]**2 + 5*theta[1]**2]
    for _ in range(n_steps):
        theta -= eta * grad_L(theta)
        losses.append(2*theta[0]**2 + 5*theta[1]**2)
    return losses

def run_nesterov(n_steps, eta, mu):
    theta = np.array([3.5, 2.5])
    v = np.zeros(2)
    losses = [2*theta[0]**2 + 5*theta[1]**2]
    for _ in range(n_steps):
        look = theta + mu * v
        v = mu * v - eta * grad_L(look)
        theta = theta + v
        losses.append(2*theta[0]**2 + 5*theta[1]**2)
    return losses

n_steps = 60
losses_gd  = run_gd(n_steps, 0.09)
losses_nes = run_nesterov(n_steps, 0.09, 0.85)
steps = np.arange(n_steps+1)

ax4.semilogy(steps, losses_gd,  'r-', lw=2, label='Gradient Descent  O(1/t)')
ax4.semilogy(steps, losses_nes, 'g-', lw=2, label='Nesterov  O(1/t²)')
ax4.set_title('Convergence: GD vs Nesterov\n(log scale — lower is faster)', fontweight='bold')
ax4.set_xlabel('Iteration t'); ax4.set_ylabel('Loss L(θ)')
ax4.legend(); ax4.grid(alpha=0.3, which='both')

# ---- Panel 5: Nesterov look-ahead visualization ----
ax5 = fig.add_subplot(gs[1, 1])
x_1d = np.linspace(-3, 4, 300)
L_1d = (x_1d - 1)**2 + 0.5
ax5.plot(x_1d, L_1d, 'b-', lw=2.5, label='$\\mathcal{L}(\\theta)$')

# Illustrate one Nesterov step
th_curr = 3.0
v_curr  = -0.4  # some momentum
mu_ill  = 0.9
look_ill = th_curr + mu_ill * v_curr
grad_ill = 2*(look_ill - 1)
th_next  = th_curr + mu_ill*v_curr - 0.3*grad_ill

ax5.scatter([th_curr], [(th_curr-1)**2+0.5], s=120, color='blue', zorder=5, label=f'θₜ = {th_curr}')
ax5.scatter([look_ill], [(look_ill-1)**2+0.5], s=120, color='orange', zorder=5, marker='^',
            label=f'Look-ahead yₜ = {look_ill:.2f}')
ax5.scatter([th_next], [(th_next-1)**2+0.5], s=120, color='green', zorder=5, marker='*',
            label=f'θₜ₊₁ = {th_next:.2f}')
ax5.scatter([1.0], [0.5], s=150, color='gold', edgecolor='black', zorder=5, marker='*', label='Optimum')

ax5.annotate('', xy=(look_ill, (look_ill-1)**2+0.5), xytext=(th_curr, (th_curr-1)**2+0.5),
             arrowprops=dict(arrowstyle='->', color='orange', lw=2))
ax5.annotate('μ·v (momentum)', xy=((th_curr+look_ill)/2, (th_curr-1)**2+0.8),
             fontsize=8, color='orange', ha='center')
ax5.annotate('', xy=(th_next, (th_next-1)**2+0.5), xytext=(look_ill, (look_ill-1)**2+0.5),
             arrowprops=dict(arrowstyle='->', color='green', lw=2))
ax5.annotate('-η∇L(yₜ)', xy=((look_ill+th_next)/2, (look_ill-1)**2+0.9),
             fontsize=8, color='green', ha='center')

ax5.set_title('Nesterov Look-Ahead Step:\n"gradient at future position"', fontweight='bold')
ax5.set_xlabel('θ'); ax5.set_ylabel('L(θ)')
ax5.legend(fontsize=8, loc='upper left'); ax5.grid(alpha=0.3)
ax5.set_xlim(-1, 4); ax5.set_ylim(0, 10)

# ---- Panel 6: Summary table ----
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
table_data = [
    ['Property', 'GD', 'Nesterov GD'],
    ['Gradient at', 'θₜ', 'θₜ + μ·vₜ (look-ahead)'],
    ['Convergence rate', 'O(1/t)', 'O(1/t²)'],
    ['Iterations needed', 'More', 'Fewer'],
    ['Optimal for convex?', 'No', 'Yes (provably)'],
    ['Extra memory', 'None', '1 vector vₜ'],
    ['Implementation', 'Simple', 'Slightly more complex'],
]
tbl = ax6.table(cellText=table_data[1:], colLabels=table_data[0],
                cellLoc='center', loc='center',
                colColours=['#DDEEFF','#FFDDDD','#DDFFDD'])
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.1, 1.8)
ax6.set_title('GD vs Nesterov — Comparison', fontweight='bold', pad=20)

plt.suptitle('Exercise 4 — Convexity & Gradient Descent with Nesterov Acceleration',
             fontsize=13, fontweight='bold')
plt.show()

---
## Summary — Exam Answer Guide

| Exercise | Sub-question | Key Answer (exam-ready) | Points |
|----------|-------------|------------------------|--------|
| **1a** | Model ansatz | $f(x) = \sum_j \theta_j B_j(x) = \mathbf{B}(x)^\top\boldsymbol{\theta}$, $\boldsymbol{\theta}\in\mathbb{R}^p$ | 1 |
| **1b** | What is fitted | Only **parameters $\boldsymbol{\theta}$**; basis functions $B_j$ are fixed a priori | 1 |
| **1c** | Normal equation | $\boldsymbol{\theta}^* = (\mathbf{B}^\top\mathbf{B})^{-1}\mathbf{B}^\top\mathbf{y}$ — closed form OLS solution | 2 |
| **2a** | Bias-Variance | $\text{MSE} = \text{Bias}^2 + \text{Var} + \sigma^2$; large $p$ → low bias, high variance | 2 |
| **2b** | Reduce variance | L2 Ridge, L1 Lasso, Early Stopping, Bagging, reduce $p$, Dropout | 3 |
| **2c** | Zero variance model | Constant $\hat{f}(x) = \bar{y}$ — zero variance but maximum bias, useless | 1 |
| **3a** | Purpose of regularization | Prevent overfitting; add penalty $\lambda\mathcal{R}(\boldsymbol{\theta})$ to loss | 1 |
| **3b** | Tikhonov regularization | $\min_f \frac{1}{n}\sum \ell(f(x_i),y_i) + \lambda\|\Gamma f\|^2$; Ridge if $\Gamma=I$ | 3 |
| **3c** | Early stopping | Stop at validation minimum; acts as L2 regularizer $\lambda\propto 1/t$ | 1 |
| **4a** | Convexity | Local min = Global min for convex $\mathcal{L}$; gradient descent converges to global opt. | 2 |
| **4b** | Nesterov pseudo-code | Look-ahead: $y_t = \theta_t + \mu v_t$; gradient at $y_t$; $O(1/t^2)$ convergence | 3 |
| | **Total** | | **20** |